In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, classification_report,
    ConfusionMatrixDisplay
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

In [3]:
df = pd.read_csv('engine1.csv')

print(f'Shape: {df.shape}')
df.head()

Shape: (32593, 32)


,num_of_prev_attempts,studied_credits,clicks_week1,clicks_week2,clicks_week3,click_trend,module_presentation_length,days_since_last_click,unique_resources,registered_late,...,region_North Western Region,region_Scotland,region_South East Region,region_South Region,region_South West Region,region_Wales,region_West Midlands Region,region_Yorkshire Region,disability_N,disability_Y
0,0,240,183,20,100,-83,268,15,55,0,...,0,0,0,0,0,0,0,0,1,0
1,0,60,175,66,37,-138,268,29,84,0,...,0,1,0,0,0,0,0,0,1,0
2,0,60,107,72,0,-107,268,256,22,0,...,1,0,0,0,0,0,0,0,0,1
3,0,60,22,154,108,86,268,4,82,0,...,0,0,1,0,0,0,0,0,1,0
4,0,60,61,116,37,-24,268,21,66,0,...,0,0,0,0,0,0,1,0,1,0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 32 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   num_of_prev_attempts         32593 non-null  int64
 1   studied_credits              32593 non-null  int64
 2   clicks_week1                 32593 non-null  int64
 3   clicks_week2                 32593 non-null  int64
 4   clicks_week3                 32593 non-null  int64
 5   click_trend                  32593 non-null  int64
 6   module_presentation_length   32593 non-null  int64
 7   days_since_last_click        32593 non-null  int64
 8   unique_resources             32593 non-null  int64
 9   registered_late              32593 non-null  int64
 10  unregistered_early           32593 non-null  int64
 11  imd_band_enc                 32593 non-null  int64
 12  education_enc                32593 non-null  int64
 13  age_enc                      32593 non-null  i

In [5]:
df.isnull().sum()

num_of_prev_attempts           0
studied_credits                0
clicks_week1                   0
clicks_week2                   0
clicks_week3                   0
click_trend                    0
module_presentation_length     0
days_since_last_click          0
unique_resources               0
registered_late                0
unregistered_early             0
imd_band_enc                   0
education_enc                  0
age_enc                        0
at_risk                        0
gender_F                       0
gender_M                       0
region_East Anglian Region     0
region_East Midlands Region    0
region_Ireland                 0
region_London Region           0
region_North Region            0
region_North Western Region    0
region_Scotland                0
region_South East Region       0
region_South Region            0
region_South West Region       0
region_Wales                   0
region_West Midlands Region    0
region_Yorkshire Region        0
disability

In [6]:
X = df.drop(columns=['at_risk'])
y = df['at_risk']

(F1 = 2 * {Precision*Recall}/{Precision + Recall}

In [7]:
def evaluate_model(name, model, X, y, cv):
    pipeline = ImbPipeline(steps=[('scaler',  StandardScaler()),('smote',   SMOTE(random_state=42)),('model',   model)])
# if there are missing values we can use ('imputer', SimpleImputer(strategy='median'))
    fold_metrics = {'precision': [], 'recall': [], 'f1': [], 'roc_auc': []}

    X_arr = X.values
    y_arr = y.values

    for foldindex, (trainindex, valindex) in enumerate(cv.split(X_arr, y_arr), 1):
        X_train, X_val = X_arr[trainindex], X_arr[valindex]
        y_train, y_val = y_arr[trainindex], y_arr[valindex]

        pipeline.fit(X_train, y_train)
        y_pred      = pipeline.predict(X_val)
        y_pred_prob = pipeline.predict_proba(X_val)[:, 1]

        fold_metrics['precision'].append(precision_score(y_val, y_pred, zero_division=0))
        fold_metrics['recall'].append(recall_score(y_val, y_pred, zero_division=0))
        fold_metrics['f1'].append(f1_score(y_val, y_pred, zero_division=0))
        fold_metrics['roc_auc'].append(roc_auc_score(y_val, y_pred_prob))

    results = {metric: np.array(vals) for metric, vals in fold_metrics.items()}
    print(f'\n')
    print(name)
    print('   Metric       Fold1   Fold2   Fold3   Fold4   Fold5    Mean    Std')
    for metric, vals in results.items():
        fold_str = '  '.join([f'{v:.4f}' for v in vals])
        print(f'  {metric.capitalize():<12}  {fold_str}  {vals.mean():.4f}  {vals.std():.4f}')

    return {metric: (vals.mean(), vals.std()) for metric, vals in results.items()}


### Models
- min_samples_leaf is used to control under or overfitting
- n_jobs controls number of cpu cores used -1 means all possible
- n_estimators
  - More Trees = Higher Stability: Increasing the number of trees generally improves the model's accuracy and smooths out variance. It helps prevent the model from being overly sensitive to specific noise in the training set.
  - Diminishing Returns: Performance does not scale linearly forever. Once the forest reaches a certain size , adding more trees will stop improving accuracy. The trees begin to capture redundant information.
  - No Overfitting Risk: Unlike algorithms like Gradient Boosting, adding more estimators to a Random Forest does not cause overfitting. You can safely use a very high number of trees without hurting performance, though you will waste computer processing power.
   - Computational Cost: The primary drawback of a massive n_estimators value is resource consumption. More trees mean longer training times and higher memory usage during both training and inference
- learning_rate In XGBoost, the learning rate scales the contribution of each newly added tree to prevent overfitting
- max_depth
-  - Smaller max_depth (e.g., 3-5): Creates simpler models that are less likely to overfit. However, if set too low, the model might underfit and fail to capture complex data patterns.
   - Larger max_depth (e.g., 10+): Allows the tree to learn highly intricate rules and capture deeper interactions in your dataset. But it significantly increases the risk of overfitting.
   - Resource Usage: Deeper trees consume memory aggressively.
- subsample is a hyperparameter that controls the fraction of training instances randomly sampled to build each tree. Setting subsample=0.8 means each tree uses a random 80% of the dataset. This introduces randomness, prevents overfitting, and reduces the variance of the overall
- colsample_bytree
  - Prevents Overfitting: By forcing the model to rely on different subsets of features for different trees, it stops the model from memorizing the data or relying too heavily on specific dominant features.
  - Speed: Using a lower colsample_bytree can also speed up training time since fewer features need to be evaluated at each split.
- Setting eval_metric='logloss' directs machine learning algorithmsto use Binary Logarithmic Loss. It heavily penalizes incorrect predictions made with high confidence, effectively teaching the model the "cost" of being wrong (try rmse later)

In [8]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000,solver='lbfgs',random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200,max_depth=None,min_samples_leaf=2,n_jobs=-1,random_state=42),
    'XGBoost': XGBClassifier(n_estimators=200,learning_rate=0.1,max_depth=10,subsample=0.8,
        colsample_bytree=0.8,eval_metric='logloss',n_jobs=-1,random_state=42
    )
}
#rmse gives same accuracy values
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

XGBClassifier since its discrete values

In [9]:
all_results = {}

for model_name, model in models.items():
    all_results[model_name] = evaluate_model(model_name,model,X,y,skf)



Logistic Regression
   Metric       Fold1   Fold2   Fold3   Fold4   Fold5    Mean    Std
  Precision     0.9838  0.9837  0.9858  0.9826  0.9841  0.9840  0.0011
  Recall        0.8672  0.8600  0.8701  0.8684  0.8643  0.8660  0.0036
  F1            0.9219  0.9177  0.9244  0.9219  0.9203  0.9212  0.0022
  Roc_auc       0.9564  0.9558  0.9571  0.9605  0.9570  0.9574  0.0016


Random Forest
   Metric       Fold1   Fold2   Fold3   Fold4   Fold5    Mean    Std
  Precision     0.9807  0.9845  0.9814  0.9830  0.9826  0.9824  0.0013
  Recall        0.8704  0.8649  0.8736  0.8724  0.8684  0.8699  0.0031
  F1            0.9223  0.9208  0.9244  0.9244  0.9219  0.9228  0.0014
  Roc_auc       0.9590  0.9591  0.9605  0.9650  0.9600  0.9607  0.0022


XGBoost
   Metric       Fold1   Fold2   Fold3   Fold4   Fold5    Mean    Std
  Precision     0.9725  0.9778  0.9708  0.9702  0.9683  0.9719  0.0032
  Recall        0.8745  0.8713  0.8780  0.8814  0.8713  0.8753  0.0039
  F1            0.9209  0.9215  0.9

mlogloss or logloss
XGBoost
   Metric       Fold1   Fold2   Fold3   Fold4   Fold5    Mean    Std
  Precision     0.9725  0.9778  0.9708  0.9702  0.9683  0.9719  0.0032
  Recall        0.8745  0.8713  0.8780  0.8814  0.8713  0.8753  0.0039
  F1            0.9209  0.9215  0.9220  0.9237  0.9172  0.9211  0.0021
  Roc_auc       0.9582  0.9584  0.9601  0.9632  0.9576  0.9595  0.0020



In [10]:
all_results

{'Logistic Regression': {'precision': (0.98401983528607,
   0.0010548177213756341),
  'recall': (0.8659925994109047, 0.0035635380651170044),
  'f1': (0.9212377123520408, 0.0022014273440245473),
  'roc_auc': (0.957353968333447, 0.0016275342919092734)},
 'Random Forest': {'precision': (0.9824173217999637, 0.0013061177027215494),
  'recall': (0.8699442633951829, 0.003092316964828413),
  'f1': (0.9227613910527458, 0.0014148101230316013),
  'roc_auc': (0.9607062475644448, 0.0022125656961945817)},
 'XGBoost': {'precision': (0.9719456734777724, 0.003230380140185719),
  'recall': (0.8752906849606068, 0.003943802038875981),
  'f1': (0.9210798143749311, 0.002134762537310554),
  'roc_auc': (0.9595017929318634, 0.002040161321521957)}}

In [11]:
metrics_order = ['precision', 'recall', 'f1', 'roc_auc']

rows = []
for model_name, metrics_dict in all_results.items():
    row = {'Model': model_name}
    for metric in metrics_order:
        mean, std = metrics_dict[metric]
        row[f'{metric} (mean)'] = round(mean, 4)
        row[f'{metric} (±std)'] = round(std, 4)
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index('Model')
print(summary_df)

                     precision (mean)  precision (±std)  recall (mean)  \
Model                                                                    
Logistic Regression            0.9840            0.0011         0.8660   
Random Forest                  0.9824            0.0013         0.8699   
XGBoost                        0.9719            0.0032         0.8753   

                     recall (±std)  f1 (mean)  f1 (±std)  roc_auc (mean)  \
Model                                                                      
Logistic Regression         0.0036     0.9212     0.0022          0.9574   
Random Forest               0.0031     0.9228     0.0014          0.9607   
XGBoost                     0.0039     0.9211     0.0021          0.9595   

                     roc_auc (±std)  
Model                                
Logistic Regression          0.0016  
Random Forest                0.0022  
XGBoost                      0.0020  


In [12]:
for model_name, metrics_dict in all_results.items():
    print(f'\n  {model_name}')
    for metric in metrics_order:
        mean, std = metrics_dict[metric]
        print(f'    {metric}: {mean:.4f} ± {std:.4f}')


  Logistic Regression
    precision: 0.9840 ± 0.0011
    recall: 0.8660 ± 0.0036
    f1: 0.9212 ± 0.0022
    roc_auc: 0.9574 ± 0.0016

  Random Forest
    precision: 0.9824 ± 0.0013
    recall: 0.8699 ± 0.0031
    f1: 0.9228 ± 0.0014
    roc_auc: 0.9607 ± 0.0022

  XGBoost
    precision: 0.9719 ± 0.0032
    recall: 0.8753 ± 0.0039
    f1: 0.9211 ± 0.0021
    roc_auc: 0.9595 ± 0.0020
